# Ingestion Bronze

L'objectif est d'ingérer les données brutes d'InduSense dans un Bronze dataset, en partant des cartes d'identité du Sprint 1.

Ce qui est demandé :
- une table par source dans PostgreSQL ;
- toutes les colonnes sources à l'identique ;
- après chaque colonne, une colonne avec son type ;
- une colonne `hash` avec l'empreinte des colonnes sources de la ligne ;
- les données des opérateurs anonymisées (sha256 + sel).

**Entrée** : `datas/releves_incidents.csv`, `datas/telemetry.csv`, `datas/machine.sql.txt`
**Sortie** : les tables `bronze.raw_*` dans PostgreSQL

## Observations sur les données

- `releves_incidents` contient **15 noms** d'opérateurs mais seulement **10 badges**. 5 badges (`OP1001`, `OP1003`, `OP1006`, `OP1007`, `OP1008`)
  sont associés à deux noms différents. C'est une incohérence, qui ne sera plus visible une fois les données anonymisées.
- `operator_name` et `operator_badge` permettent d'identifier une personne : ce sont des données personnelles.
- `comment` contient des libellés techniques (« chauffe anormale », « alarme capteur »…), aucun nom n'y apparaît.
  Mais c'est un champ libre, un nom pourrait y être saisi.
- `telemetry` n'a aucune ligne strictement identique, mais **1 346 lignes** ont la même machine et la même heure qu'une autre ligne,
  avec des valeurs légèrement différentes.
- `machine.sql.txt` est un script PostgreSQL qui contient **2 tables** : `machine` et `maintenance`.

## Où stocker le Bronze

Choix : **PostgreSQL**, avec un schéma `bronze` et une table par table source :
`bronze.raw_releves_incidents`, `bronze.raw_telemetry`, `bronze.raw_machine`, `bronze.raw_maintenance`.

Pourquoi :
- les étapes suivantes (Silver, Gold) seront aussi en base ;
- on peut interroger les données en SQL et gérer qui y a accès ;
- le docker-compose fourni lance PostgreSQL et pgAdmin.

Le fichier SQL contient deux tables : il donne donc deux tables Bronze.

## Garder les colonnes à l'identique

Si pandas devine les types à la lecture, il modifie certaines valeurs : `4` peut devenir `4.0`,
les cases vides deviennent `NaN`, etc. Les données ne seraient plus identiques à la source.

Tout est donc lu **en texte** (`dtype=str`) et stocké en `TEXT`.
Le type n'est pas perdu : il est indiqué dans la colonne d'à côté. La conversion dans les vrais types se fera en Silver.

## La colonne de type

Après chaque colonne, une colonne `<colonne>_type` est ajoutée, par exemple `temperature_c` puis `temperature_c_type` = `FLOAT`.

Le type vient des fonctions du Sprint 1 :
- pour les CSV, `infer_type` devine le type à partir des valeurs ;
- pour le SQL, `sql_column_type` lit le type déclaré dans le `CREATE TABLE`.

Le vocabulaire est donc le même que dans les cartes d'identité (`IDENTITY`, `INTEGER`, `FLOAT`, `DATE`…).

Un type par colonne (le même sur toutes les lignes), car la consigne parle du type de la colonne.
Un type par valeur permettrait de repérer une valeur atypique dans une colonne, mais c'est plus lourd.

## La colonne hash

Le hash sert à :
- vérifier qu'une ligne n'a pas été modifiée ;
- repérer des lignes identiques ;
- savoir si une ligne a changé quand on relance l'ingestion.

Algorithme : **SHA-256**, car MD5 et SHA-1 ne sont plus considérés comme sûrs.

Le hash porte sur les colonnes sources uniquement (pas sur les colonnes `_type`), toujours dans le même ordre
et avec le même séparateur, pour que la même ligne donne toujours le même hash.

Il est calculé **après l'anonymisation**. Calculé sur le vrai nom, il permettrait à quelqu'un qui a accès à la table (et connaît donc
les autres colonnes de la ligne) d'essayer les 15 noms un par un jusqu'à retrouver le même hash, et donc la personne.

Limite : le hash ne repère pas les 1 346 doublons de la télémétrie, parce que leurs valeurs ne sont pas exactement les mêmes.
Il faudra les traiter en Silver en se basant sur la machine et l'heure.

## L'anonymisation des opérateurs

Prédire une panne ne nécessite pas de savoir quel opérateur a fait le relevé. `operator_name` et `operator_badge`
sont donc remplacés par `sha256(sel + valeur)`.

- Les colonnes sont conservées (la consigne demande toutes les colonnes), mais l'identité n'est plus lisible.
- Le même opérateur a toujours le même code, donc on peut encore compter ses incidents sans connaître son identité.
- Un **sel** est ajouté car sans sel, les badges vont de `OP1001` à `OP1010` : il suffirait de hacher ces 10 valeurs pour tout retrouver.
- Le sel est dans le fichier `.env`, non versionné, et diffère de celui de l'exemple.

C'est fait dès le Bronze pour que les noms n'arrivent jamais dans la base.

Limites :
- techniquement c'est une pseudonymisation : avec le sel, on pourrait refaire le lien. Il faut donc protéger le sel ;
- après le hachage, on ne verra plus les badges partagés entre deux noms. Il faut les noter avant.

## Le fichier SQL

Le script n'est pas exécuté dans PostgreSQL : il est lu avec **sqlglot**, comme au Sprint 1.

Exécuté, il ferait créer à PostgreSQL des tables avec des types et remplirait lui-même certaines colonnes (`DEFAULT NOW()`) :
ce ne seraient plus les données brutes. Et exécuter un script reçu tel quel n'est pas prudent.

Les colonnes `is_active`, `created_at` et `updated_at` sont déclarées mais n'ont aucune valeur dans les `INSERT`.

## Relancer l'ingestion

À chaque exécution, les tables sont **remplacées** : relancer donne exactement le même résultat, sans doublons.

Le guide propose aussi d'ajouter des colonnes de suivi (`_ingested_at`, `_source_file`, `_batch_id`) et une table qui garde
l'historique des chargements. Ce n'est pas demandé dans la consigne, donc pas fait pour l'instant.

## Les librairies

| Besoin | Librairie | Pourquoi |
|---|---|---|
| Lire les CSV et préparer les tables | pandas | déjà utilisée au Sprint 1, et `to_sql` écrit directement dans la base |
| Lire le script SQL | sqlglot | déjà utilisée au Sprint 1 |
| Se connecter à PostgreSQL | SQLAlchemy + psycopg | `to_sql` a besoin de SQLAlchemy, et psycopg 3 est le pilote PostgreSQL actuel |
| Lire le `.env` | python-dotenv | simple et léger |
| Calculer le SHA-256 | hashlib | déjà inclus dans Python |

Écartés : polars (pas utile pour 135 000 lignes), psycopg2 (ancienne version) et pydantic-settings (trop complet pour un notebook).

## Vérifications

- chaque table a le même nombre de lignes que sa source ;
- on retrouve toutes les colonnes, dans le même ordre, chacune suivie de sa colonne `_type` ;
- aucun nom ni badge en clair dans la base ;
- le hash d'une ligne recalculé à la main est identique à celui de la table ;
- les fichiers de `datas/` n'ont pas été modifiés.

## À quoi ressemblera une table

Exemple pour `bronze.raw_releves_incidents` :

| incident_id | incident_id_type | date | date_type | … | operator_name | operator_name_type | … | hash |
|---|---|---|---|---|---|---|---|---|
| INC-000001 | IDENTITY | 2025-06-01 | DATE | … | `9f2c…` | TEXT | … | `5b8d…` |

## Étapes

1. Mettre en place PostgreSQL : `docker-compose.yml`, `.env` avec un sel propre au projet, `.env` dans le `.gitignore`.
2. Déplacer les fonctions de typage du Sprint 1 dans `src/indusense/` pour les utiliser dans les deux notebooks.
3. Écrire l'ingestion : lecture, colonnes de type, anonymisation, hash, chargement.
4. Faire les vérifications.

## Implémentation

### 1. Imports et configuration

Le sel et les accès à la base sont lus dans le `.env` (jamais affichés).
`PERSONAL_COLUMNS` liste les colonnes à pseudonymiser.

In [ ]:
import hashlib
import json
import os
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import URL, create_engine, text
from sqlalchemy.types import TEXT

from indusense.profiling import detect_charset, detect_separator, infer_type, parse_sql_tables

DATA_DIR = Path("datas")
PERSONAL_COLUMNS = ["operator_name", "operator_badge"]

load_dotenv()
SALT = os.environ["ANONYMISATION_SALT"]

### 2. Lecture brute des sources

`read_csv_source` lit le CSV deux fois : une fois **en texte** pour garder les valeurs à l'identique, une fois avec les types
de pandas pour que `infer_type` devine le type de chaque colonne (comme au Sprint 1).

`read_sql_source` lit le script avec sqlglot en mode `raw=True` : les valeurs restent le texte écrit dans le script.
Les colonnes déclarées dans le `CREATE TABLE` mais absentes des `INSERT` (`is_active`, `created_at`, `updated_at`) sont gardées, vides.

**Entrée** : chemin du fichier
**Sortie** : `(DataFrame en texte, {colonne: type})` pour un CSV, `{table: (DataFrame, {colonne: type})}` pour le SQL

In [ ]:
def read_csv_source(path):
    charset = detect_charset(path)
    sep = detect_separator(path, charset)
    raw = pd.read_csv(path, sep=sep, encoding=charset, dtype=str, keep_default_na=False)
    typed = pd.read_csv(path, sep=sep, encoding=charset)
    return raw, {col: infer_type(col, typed[col]) for col in typed.columns}


def read_sql_source(path):
    tables = parse_sql_tables(Path(path).read_text(encoding=detect_charset(path)), raw=True)
    result = {}
    for name, (df, types) in tables.items():
        df = df.reindex(columns=list(types)).astype(object)  # toutes les colonnes déclarées, dans l'ordre du CREATE TABLE
        result[name] = (df.where(df.notna(), None), types)   # NULL reste None (et non NaN)
    return result

### 3. Pseudonymisation et hash

- `pseudonymize` : `sha256(sel + valeur)` ; une valeur vide reste vide.
- `row_hash` : SHA-256 des valeurs de la ligne, sérialisées en JSON (ordre des colonnes fixe, `None` distinct de `""`).

**Entrée** : une valeur / les valeurs d'une ligne
**Sortie** : une empreinte hexadécimale de 64 caractères

In [ ]:
def pseudonymize(value):
    if value is None or value == "":
        return value
    return hashlib.sha256((SALT + value).encode("utf-8")).hexdigest()


def row_hash(values):
    return hashlib.sha256(json.dumps(list(values), ensure_ascii=False).encode("utf-8")).hexdigest()

### 4. Construction de la table Bronze

Dans l'ordre : pseudonymisation des colonnes personnelles, calcul du `hash` sur les colonnes sources,
puis ajout de la colonne `<colonne>_type` après chaque colonne.

**Entrée** : DataFrame en texte, `{colonne: type}`
**Sortie** : DataFrame Bronze (`col1`, `col1_type`, `col2`, `col2_type`, …, `hash`)

In [ ]:
def build_bronze_table(df, types):
    df = df.copy()
    for col in PERSONAL_COLUMNS:
        if col in df.columns:
            df[col] = df[col].map(pseudonymize)
    hashes = [row_hash(row) for row in df.itertuples(index=False)]
    columns = {}
    for col in df.columns:
        columns[col] = df[col]
        columns[f"{col}_type"] = types[col]
    columns["hash"] = hashes
    return pd.DataFrame(columns)

### 5. Chargement dans PostgreSQL

`get_engine` crée la connexion à partir du `.env` et le schéma `bronze` s'il n'existe pas.
`load_table` remplace la table (relancer donne le même résultat) ; toutes les colonnes sont en `TEXT`.

**Entrée** : DataFrame Bronze, nom de la table
**Sortie** : la table `bronze.<nom>` dans la base

In [ ]:
def get_engine():
    url = URL.create("postgresql+psycopg", username=os.environ["DB_USER"], password=os.environ["DB_PASSWORD"],
                     host="localhost", port=int(os.environ["DB_PORT"]), database=os.environ["DB_NAME"])
    engine = create_engine(url)
    with engine.begin() as conn:
        conn.execute(text("CREATE SCHEMA IF NOT EXISTS bronze"))
    return engine


def load_table(df, name, engine):
    df.to_sql(name, engine, schema="bronze", if_exists="replace", index=False,
              chunksize=10_000, dtype={col: TEXT() for col in df.columns})

### 6. Auto-contrôle

Tests rapides sur un mini-tableau, sans base de données.

In [ ]:
mini = pd.DataFrame({"incident_id": ["INC-1", "INC-2"], "operator_name": ["Alice Martin", ""]})
bronze = build_bronze_table(mini, {"incident_id": "IDENTITY", "operator_name": "TEXT"})

assert list(bronze.columns) == ["incident_id", "incident_id_type", "operator_name", "operator_name_type", "hash"]
assert bronze["operator_name"][0] == pseudonymize("Alice Martin") != "Alice Martin"  # même valeur -> même code
assert bronze["operator_name"][1] == ""                                              # vide reste vide
assert bronze["hash"][0] == row_hash(["INC-1", pseudonymize("Alice Martin")])         # hash après pseudonymisation
assert row_hash([None]) != row_hash([""])
print("OK")

### 7. Ingestion

Une table `bronze.raw_<source>` par CSV, et une par table du script SQL.

Les empreintes SHA-256 des fichiers de `datas/` sont relevées avant, pour la vérification finale.

In [ ]:
def file_checksums():
    return {p.name: hashlib.sha256(p.read_bytes()).hexdigest() for p in sorted(DATA_DIR.iterdir()) if p.is_file()}


checksums_before = file_checksums()

sources = {}
for path in sorted(DATA_DIR.glob("*.csv")):
    sources[path.stem] = read_csv_source(path)
for path in sorted(DATA_DIR.glob("*.sql*")):
    sources.update(read_sql_source(path))

engine = get_engine()
for name, (df, types) in sources.items():
    table = build_bronze_table(df, types)
    load_table(table, f"raw_{name}", engine)
    print(f"bronze.raw_{name} : {len(table)} lignes, {table.shape[1]} colonnes")

### 8. Vérifications

Les tables sont relues **depuis la base** et comparées aux sources :
- même nombre de lignes que la source ;
- toutes les colonnes, dans le même ordre, chacune suivie de sa colonne `_type` (avec le bon type), puis `hash` ;
- aucun nom ni badge en clair : chaque valeur est une empreinte de 64 caractères hexadécimaux ;
- le `hash` de **chaque** ligne, recalculé à partir des valeurs lues en base, est identique à celui stocké ;
- les fichiers de `datas/` ont la même empreinte qu'avant l'ingestion.

**Entrée** : les sources lues à l'étape 7, la base
**Sortie** : un message OK par table, ou une erreur qui indique ce qui ne va pas

In [ ]:
def check_table(name, df, types, engine):
    db = pd.read_sql_table(f"raw_{name}", engine, schema="bronze").astype(object)
    db = db.where(db.notna(), None)  # NULL lu en base -> None, comme à l'ingestion

    assert len(db) == len(df), f"{name} : {len(db)} lignes en base, {len(df)} dans la source"

    expected = [c for col in df.columns for c in (col, f"{col}_type")] + ["hash"]
    assert list(db.columns) == expected, f"{name} : colonnes différentes de la source"
    for col in df.columns:
        assert (db[f"{col}_type"] == types[col]).all(), f"{name}.{col}_type : type incorrect"

    for col in PERSONAL_COLUMNS:
        if col in df.columns:
            clear_values = set(df[col]) - {""}
            assert not clear_values & set(db[col]), f"{name}.{col} : valeur en clair dans la base"
            assert db[col].str.fullmatch(r"[0-9a-f]{64}").all(), f"{name}.{col} : valeur non pseudonymisée"

    recomputed = [row_hash(row) for row in db[list(df.columns)].itertuples(index=False)]
    assert recomputed == list(db["hash"]), f"{name} : hash différent"
    print(f"bronze.raw_{name} : OK ({len(db)} lignes)")


for name, (df, types) in sources.items():
    check_table(name, df, types, engine)

assert file_checksums() == checksums_before, "un fichier de datas/ a été modifié"
print("datas/ : fichiers inchangés")

## Points à confirmer

- Un type par colonne, ou un type par valeur ?
- Deux tables Bronze pour le fichier SQL, est-ce que ça convient ?
- Les colonnes SQL sans valeur : les garder vides ou ne pas les créer ?
- Les colonnes de suivi du guide sont-elles attendues dès maintenant ?
- Le guide met l'anonymisation en Silver, la consigne en Bronze : choix retenu, le Bronze.
- Où noter les badges partagés avant de les anonymiser ?